# Prepare GND Data

In [1]:
import pandas as pd
import re
import webbrowser
from tqdm import tqdm
import numpy as np
import glob

# Import metadata

In [26]:
# Read in: previously chosen relevant occupations together with their "SachgruppenID"
df = pd.read_csv("../data/occupations_annotated.csv", sep=";", encoding="utf-8")

# Read in root of GND Bulk download link für occupations
with open("../data/linkroot.txt", "r") as f:
    linkroot = f.read()


# Get the Link for each occupation

In [13]:
# Create link for each occupation

links = []
for gnd_id in df.GND_ID:
    link = re.sub("insert_gnd_id_here", gnd_id, linkroot)
    links.append(link)
df["Download_Link"] = links

#df.to_excel("../data/occupations_annotated_with_link.xlsx")

# Download data for each Link
(Attention: This opens a lot of windows in your Browser to download the data, data will be stored in "Download" Folder of your device afterwards. )

In [37]:
for link in tqdm(links):
    webbrowser.open(link)

100%|███████████████████████████████████████████| 48/48 [00:05<00:00,  8.06it/s]


# Transform data to csv
(In between: copy data from Download folder in respective folder in datafolder)

In [4]:
# Define functions for reading in raw json and output raw csv
def read_in_json_and_transform_to_csv(download_id):
    data_beruf = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Reading JSONL"):
            record = json.loads(line)
            data_beruf.append(record)
    
    seen = set()
    data_gnd = []
    
    for item in data_beruf:
        gnd_id = item.get("gndIdentifier")
        if gnd_id and gnd_id not in seen:
            seen.add(gnd_id)
            data_gnd.append(item)
    
    beruf_df = pd.DataFrame(data_beruf)
    
    beruf_df.to_csv(f"../data/GND_raw/csv/{download_id}.csv", sep=";", encoding="utf-8")
    return(beruf_df)

In [6]:
# Define Functions for transforming raw csv in format for our analysis

def transform_column_type_1_value(row, column):
    value = []
    for entry in row[column]:
        value.append(entry["label"])
    value = ", ".join(value)
    return value
    
def convert_to_year(date):
    if pd.isna(date):
        return date
    else:
        date_parts = date.split('-')
        if date_parts[0] == '':
            bc = True
            date = date_parts[1]
        else:
            bc = False
            date = date_parts[0]
        date = date.replace('X', '0')
        match = re.findall(r"\d+", date)
        if match:
            date = int(match[0])
            if bc:
                date = -date
            return date
        else:
            print(date)
            return None

def transform_df(beruf_df):
    relevant_columns_type_1 = ["professionOrOccupation", "gender", "geographicAreaCode", "placeOfBirth"]
    relevant_columns_type_2 = ["dateOfBirth", "dateOfDeath", "periodOfActivity"]

    data = {"gndIdentifier": [], "preferredName": [], "professionOrOccupation": [], 
            "gender": [], "geographicAreaCode": [], "placeOfBirth": [], "dateOfBirth": [], 
           "dateOfDeath": [], "periodOfActivity": []}
    
    
    for _,row in beruf_df.iterrows():
        data["gndIdentifier"].append(row["gndIdentifier"])
        data["preferredName"].append(row["preferredName"])
        
        for column in relevant_columns_type_1:
            if column not in beruf_df.columns:
                data[column].append(np.nan)
            else:
                if type(row[column])!=list:
                    data[column].append(np.nan)
                else:
                    data[column].append(transform_column_type_1_value(row, column))
                
        for column in relevant_columns_type_2:
            if column not in beruf_df.columns:
                data[column].append(np.nan)
            else:
                if type(row[column])!=list:
                    data[column].append(np.nan)
                else:
                    if len(row[column])>1:
                        data[column].append(row[column])
                    else:
                        data[column].append(convert_to_year(row[column][0]))
            
    beruf_df_transformed = pd.DataFrame(data)
    return beruf_df_transformed

In [8]:
# Apply Functions: Read in all files and transform 
all_data = []

file_paths = glob.glob("../data/GND_raw/json/*.jsonl")
for file_path in file_paths:
    download_id = file_path.split("-")[3].split(".")[0]
    beruf_df = read_in_json_and_transform_to_csv(download_id)
    transformed_df = transform_df(beruf_df)
    transformed_df.to_csv(f"../data/GND_raw/csv_transformed/{download_id}.csv", sep=";", encoding="utf-8")
    all_data.append(transformed_df)

Reading JSONL: 282it [00:00, 36638.28it/s]
Reading JSONL: 2250it [00:00, 32201.35it/s]
Reading JSONL: 213it [00:00, 44467.01it/s]
Reading JSONL: 98it [00:00, 26474.42it/s]
Reading JSONL: 1790it [00:00, 31684.11it/s]
Reading JSONL: 30it [00:00, 25105.57it/s]
Reading JSONL: 1473it [00:00, 48674.92it/s]
Reading JSONL: 461it [00:00, 11601.56it/s]
Reading JSONL: 459it [00:00, 47922.37it/s]
Reading JSONL: 848it [00:00, 44699.32it/s]
Reading JSONL: 1478it [00:00, 27301.59it/s]
Reading JSONL: 1it [00:00, 2801.81it/s]
Reading JSONL: 1432it [00:00, 46062.97it/s]
Reading JSONL: 1394it [00:00, 25021.97it/s]
Reading JSONL: 482it [00:00, 62065.35it/s]
Reading JSONL: 1865it [00:00, 36763.26it/s]
Reading JSONL: 7886it [00:00, 28733.20it/s]
Reading JSONL: 8612it [00:00, 29451.40it/s]
Reading JSONL: 1150it [00:00, 50558.68it/s]
Reading JSONL: 32it [00:00, 6403.82it/s]
Reading JSONL: 1117it [00:00, 35654.78it/s]
Reading JSONL: 782it [00:00, 17437.43it/s]
Reading JSONL: 17479it [00:00, 24800.08it/s]
Readi

ca. Ming


Reading JSONL: 886it [00:00, 30961.75it/s]
Reading JSONL: 6it [00:00, 8836.31it/s]
Reading JSONL: 541it [00:00, 48613.20it/s]
Reading JSONL: 4549it [00:00, 18681.27it/s]
Reading JSONL: 763it [00:00, 53464.99it/s]
Reading JSONL: 841it [00:00, 46211.41it/s]
Reading JSONL: 4199it [00:00, 27186.21it/s]
Reading JSONL: 779it [00:00, 42438.79it/s]
Reading JSONL: 573it [00:00, 11324.79it/s]
Reading JSONL: 619it [00:00, 48458.74it/s]
Reading JSONL: 8610it [00:00, 28566.81it/s]
Reading JSONL: 741it [00:00, 51651.59it/s]
Reading JSONL: 2441it [00:00, 32367.92it/s]
Reading JSONL: 46it [00:00, 24376.25it/s]
Reading JSONL: 24it [00:00, 20514.22it/s]
Reading JSONL: 99it [00:00, 29757.50it/s]
Reading JSONL: 17it [00:00, 15071.48it/s]
Reading JSONL: 1263it [00:00, 23515.60it/s]
Reading JSONL: 28160it [00:01, 26847.35it/s]


In [9]:
# Merging all dataframes and dropping duplicates (by gnd identifier)
merged_df = pd.concat(all_data, ignore_index=True).reset_index(drop=True)
print(len(merged_df))
merged_df = merged_df.drop_duplicates(subset=["gndIdentifier"])
print(len(merged_df))

342346
306190


# Filtering dataframe by our criteria

In [18]:
print(len(merged_df))

# Birth year filter
merged_df_filtered = merged_df[merged_df["dateOfBirth"] > 1549]
print(len(merged_df_filtered))

# Country filter
merged_df_filtered_d = merged_df_filtered[merged_df_filtered["geographicAreaCode"].str.contains("Deutschland", na=False)]
merged_df_filtered_ö = merged_df_filtered[merged_df_filtered["geographicAreaCode"].str.contains("Österreich", na=False)]
merged_df_filtered_s = merged_df_filtered[merged_df_filtered["geographicAreaCode"].str.contains("Schweiz", na=False)]

merged_filtered_new = pd.concat([merged_df_filtered_d, merged_df_filtered_ö, merged_df_filtered_s], ignore_index=True).reset_index(drop=True)
merged_filtered_new = merged_filtered_new.drop_duplicates(subset=["gndIdentifier"]).reset_index(drop=True)

print(len(merged_filtered_new))

306190
236654
79009


## Rename Columns with our terminology

In [21]:
merged_filtered_new = merged_filtered_new.rename(columns={"preferredName": "GND_name", 
                               "professionOrOccupation": "GND_occupation",
                               "gender": "GND_gender",
                               "geographicAreaCode": "GND_country",
                               "dateOfBirth": "GND_birth",
                               "dateOfDeath": "GND_death",})

## Correct wrong time data

In [23]:
# Birth year
correct_birth_years = []
correct_death_years = []

## birth year after 2020
for _, row in merged_filtered_new.iterrows():
    birth_year = row["GND_birth"]
    if birth_year > 2020:
        print(f"ERROR: {row["GND_name"]} with id {row["gndIdentifier"]} has birth year {birth_year}, corrected to NaN")
        correct_birth_years.append(np.nan)
    else:
        correct_birth_years.append(birth_year)

    death_year = row["GND_death"]
    if death_year < birth_year+1:
        print(f"ERROR: {row["GND_name"]} with id {row["gndIdentifier"]} has death_year ({death_year}) before birth_year ({birth_year}), corrected to NaN")
        correct_death_years.append(np.nan)
    # Grob Autoren rausfiltern, bei denen Todesjahr zu hoch
    elif death_year > birth_year+120:
        print(f"ERROR: {row["GND_name"]} with id {row["gndIdentifier"]} has death ({death_year}) year more than 120 years after birth year ({birth_year}), corrected to NaN")
        correct_death_years.append(np.nan)
    else:
        correct_death_years.append(death_year)

merged_filtered_new["GND_birth"] = correct_birth_years
merged_filtered_new["GND_death"] = correct_death_years

ERROR: Klaus, Albert with id 1348804343 has death_year (0.0) before birth_year (1872.0), corrected to NaN
ERROR: Platz, Friedrich Wilhelm with id 1230445641 has death_year (0.0) before birth_year (1851.0), corrected to NaN
ERROR: Breiderhoff, Karl with id 189561157 has death_year (0.0) before birth_year (1893.0), corrected to NaN
ERROR: Lennig, Friedrich with id 133246957 has death_year (1796.0) before birth_year (1796.0), corrected to NaN
ERROR: Stahn, Hannah with id 1267684607 has death_year (0.0) before birth_year (1886.0), corrected to NaN
ERROR: Mohr, Marie with id 117093300 has death_year (0.0) before birth_year (1850.0), corrected to NaN
ERROR: Schwabacher-Bleichröder, Anna with id 117333077 has death_year (1040.0) before birth_year (1870.0), corrected to NaN
ERROR: Runkel, Ferdinand with id 1014229820 has death_year (1846.0) before birth_year (1864.0), corrected to NaN
ERROR: Lange, Hellmuth with id 106272217 has death_year (0.0) before birth_year (1903.0), corrected to NaN
ERR

## Save Dataframe

In [25]:
merged_filtered_new.to_csv("../data/gnd_all.csv", sep=";", encoding="utf-8")